# 01 — train / validation split + leak-guard

Rung 01 (infra, not a scored rung). Builds the frozen **train + validation** partition (`train`, `val_id`, `val_ood`) every later experiment trains + selects checkpoints against. Baseline = `00-baseline` (zero-shot, no split).

**No local test** — the real test is the leaderboard, so we don't waste data on one. **The rule:** split by `(dataset, video_id)`, NEVER by question/frame. Hold out ONE whole `procedure_type` as `val_ood` (keep the other surgeries in train → the model learns cross-procedure; OOD is half the score). Uses BOTH batches (heico + lapchole). Writes the manifest to the shared `experiments/splits/` folder — the source of truth every notebook reloads via `frame.split.load_manifest`.

In [ ]:
# ── bootstrap ─────────────────────────────────────────
import sys, logging
from pathlib import Path

EXP_DIR = Path.cwd()
REPO = EXP_DIR
while REPO != REPO.parent and not ((REPO / ".git").exists() or (REPO / "src").is_dir()):
    REPO = REPO.parent
for p in (EXP_DIR / "_models", REPO / "src"):
    if p.is_dir():
        sys.path.insert(0, str(p))
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s %(message)s", datefmt="%H:%M:%S")

from frame.config import BaselineConfig
from frame.data import load_frame_items
from frame import split as sp

## Sibling rungs (change ONE value, rerun, bump the manifest name)
- 01a: `ood_procedure` = a different held-out procedure_type
- 01b: `val_frac` = 0.20
Each is a new `experiments/splits/frame_ood_v<N>.csv` + a RESULTS.csv row — NOT a new notebook.

In [ ]:
# ── inspect: what procedure_types exist? (choose the OOD holdout) ──────
SMOKE = False                                       # True => cap 2 questions/video (keeps ALL videos so the split still works) for a fast logic check
items = load_frame_items(BaselineConfig())          # BOTH batches: heico + lapchole
if SMOKE:
    from collections import Counter
    seen = Counter(); _smk = []
    for it in items:
        k = (it.dataset, it.video_id)
        if seen[k] < 2:
            _smk.append(it); seen[k] += 1
    items = _smk
    print(f"[SMOKE] {len(items)} items across {len(seen)} videos")
vids = sp.videos_table(items)
print(f"videos: {len(vids)}  |  questions: {len(items)}")
print(vids.groupby(["dataset", "procedure_type"]).agg(n_videos=("video_id", "count"), n_questions=("n_questions", "sum")))

In [ ]:
# ── config (inline) + build the split ──────────────────────────
# THE one lever: which whole procedure_type is held out as OOD.
# Set it from the table above. If left None, auto-pick the heico procedure with
# the FEWEST videos (smallest OOD slice) and warn — override for the real run.
OOD_PROCEDURE = None

if OOD_PROCEDURE is None:
    heico = vids[vids["dataset"] == "heico"]
    OOD_PROCEDURE = (
        heico.groupby("procedure_type")["video_id"].count().sort_values().index[0]
    )
    print(f"[auto] OOD_PROCEDURE = {OOD_PROCEDURE!r}  (override for the real split)")

cfg = sp.SplitConfig(
    ood_procedure = OOD_PROCEDURE,
    ood_dataset   = "heico",
    val_frac      = 0.15,
    seed          = 42,
    manifest_path = REPO / "experiments" / "splits" / "frame_ood_v1.csv",
)
video_split = sp.build_split(items, cfg)             # asserts no leak internally

In [ ]:
# ── report the 80/20 table + coverage + snapshot + write the manifest ──────────
sp.assert_no_leak(video_split)
sp.assert_all_matched(items, video_split)           # loud fail if any item is unmatched (silent-drop guard)

print("Scenario A — train + validation (no local test; real test = leaderboard):")
print(sp.split_summary(items, video_split).to_string(index=False))

cov = sp.per_bucket_report(items, video_split)
print("\ncapability_group × {ID,OOD} coverage (want all 5 groups on BOTH sides):")
print(cov.to_string(index=False))

# config snapshot (spec §5d) — the exact SplitConfig that produced this manifest
import yaml
snap = dict(ood_procedure=cfg.ood_procedure, ood_dataset=cfg.ood_dataset,
            val_frac=cfg.val_frac, seed=cfg.seed, smoke=SMOKE,
            manifest=str(cfg.manifest_path), sha256=sp.manifest_hash(video_split))
cfg.manifest_path.parent.mkdir(parents=True, exist_ok=True)
with open(str(cfg.manifest_path) + ".config.yaml", "w") as f:
    yaml.safe_dump(snap, f, sort_keys=False)

manifest = sp.write_manifest(video_split, vids, cfg)  # also writes the .sha256 sidecar
print(f"\nwrote {manifest} (+ .sha256 + .config.yaml)  — commit these; every experiment reloads the manifest.")

## Result
Only two roles: **train + validation** (`val_id` + `val_ood`). **No local test** — the real test is the leaderboard, so we don't waste data on one. `val_*` is small and used only to select checkpoints (by `val_ood`, so we don't overfit to cholecystectomy). For the FINAL submission model, fold validation back into train and retrain on everything.

The frozen split lives at `experiments/splits/frame_ood_v1.csv` (committed — shared source of truth). Downstream notebooks do:
```python
from frame import split as sp
video_split = sp.load_manifest(REPO / "experiments" / "splits" / "frame_ood_v1.csv")  # verifies sha256
train_items = sp.apply_split(items, video_split, "train")
val_ood     = sp.apply_split(items, video_split, "val_ood")   # the OOD selection gauge
```
**Watch the coverage table:** if any of the 5 capability groups is missing on the OOD side, that bucket is un-scoreable — broaden the OOD holdout (or the FRAME test genuinely lacks it) before trusting any fine-tuned OOD number. Record the outcome in `RESULTS.csv` and `context/01-ood-split/CONTEXT.md`.

In [ ]:
# ── (OPTIONAL, final certainty only) Leave-One-Procedure-Out k-fold ──────────
# Do NOT run this in the dev loop — it means one LoRA training run PER fold (k× GPU cost).
# Use it once near the end for a robust OOD estimate that wastes no data: each heico
# procedure is val_ood in exactly one fold and train in the others.
RUN_KFOLD = False
if RUN_KFOLD:
    for proc, fold_split in sp.kfold_lopo(items, ood_dataset="heico", seed=42):
        s = sp.split_summary(items, fold_split)
        n_ood = int(s.loc[s.split == "val_ood", "n_questions"].iloc[0])
        n_tr = int(s.loc[s.split == "train", "n_questions"].iloc[0])
        print(f"fold OOD={proc:16s}  train_q={n_tr:5d}  val_ood_q={n_ood:5d}")
        # per fold: train LoRA on 'train', eval on 'val_ood' -> collect acc_OOD, then average across folds.